# Statistical Analysis Basics
## Descriptive statistics, distributions, and hypothesis testing

## 1. Setup and Sample Data

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt

# Create sample data
np.random.seed(42)
data = np.random.normal(loc=100, scale=15, size=1000)

df = pd.DataFrame({
    'Score': data,
    'Group': np.random.choice(['A', 'B'], size=1000)
})

print("Sample data:")
print(df.head(10))

## 2. Descriptive Statistics

In [ ]:
# Basic statistics
print("Basic Statistics:")
print(f"Mean: {df['Score'].mean():.2f}")
print(f"Median: {df['Score'].median():.2f}")
print(f"Mode: {df['Score'].mode()[0]:.2f}")
print(f"Std Dev: {df['Score'].std():.2f}")
print(f"Variance: {df['Score'].var():.2f}")
print(f"Skewness: {df['Score'].skew():.2f}")
print(f"Kurtosis: {df['Score'].kurtosis():.2f}")

In [ ]:
# Describe method
print("\nDescriptive Summary:")
print(df['Score'].describe())

# Quartiles and IQR
q1 = df['Score'].quantile(0.25)
q3 = df['Score'].quantile(0.75)
iqr = q3 - q1

print(f"\nQuartiles:")
print(f"Q1 (25%): {q1:.2f}")
print(f"Q3 (75%): {q3:.2f}")
print(f"IQR: {iqr:.2f}")

## 3. Distribution Analysis

In [ ]:
# Histogram
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.hist(df['Score'], bins=30, edgecolor='black', alpha=0.7)
plt.xlabel('Score')
plt.ylabel('Frequency')
plt.title('Distribution of Scores')

# Box plot
plt.subplot(1, 2, 2)
plt.boxplot(df['Score'])
plt.ylabel('Score')
plt.title('Box Plot of Scores')
plt.tight_layout()
plt.show()

print("Distribution plots created.")

## 4. Correlation and Covariance

In [ ]:
# Create multi-variable data
df_corr = pd.DataFrame({
    'Height': [170, 172, 168, 175, 180, 165],
    'Weight': [70, 75, 68, 80, 85, 65],
    'Age': [25, 30, 22, 35, 28, 20]
})

# Pearson correlation
print("Correlation Matrix:")
corr_matrix = df_corr.corr()
print(corr_matrix)

# Covariance
print("\nCovariance Matrix:")
cov_matrix = df_corr.cov()
print(cov_matrix)

In [ ]:
# Correlation coefficient with p-value
height = df_corr['Height']
weight = df_corr['Weight']

corr, p_value = stats.pearsonr(height, weight)
print(f"Pearson correlation: {corr:.4f}")
print(f"P-value: {p_value:.4f}")

# Spearman correlation
spearman_corr, spearman_p = stats.spearmanr(height, weight)
print(f"\nSpearman correlation: {spearman_corr:.4f}")
print(f"P-value: {spearman_p:.4f}")

## 5. Hypothesis Testing

In [ ]:
# Create two groups
group_a = df[df['Group'] == 'A']['Score']
group_b = df[df['Group'] == 'B']['Score']

print(f"Group A - Mean: {group_a.mean():.2f}, Std: {group_a.std():.2f}")
print(f"Group B - Mean: {group_b.mean():.2f}, Std: {group_b.std():.2f}")

# Independent samples t-test
t_stat, t_pval = stats.ttest_ind(group_a, group_b)
print(f"\nIndependent t-test:")
print(f"t-statistic: {t_stat:.4f}")
print(f"p-value: {t_pval:.4f}")
print(f"Significant at α=0.05: {'Yes' if t_pval < 0.05 else 'No'}")

In [ ]:
# Chi-square test
contingency_table = pd.crosstab(df['Group'], pd.cut(df['Score'], bins=2))
print("Contingency Table:")
print(contingency_table)

chi2, chi2_pval, dof, expected = stats.chi2_contingency(contingency_table)
print(f"\nChi-square test:")
print(f"Chi-square statistic: {chi2:.4f}")
print(f"p-value: {chi2_pval:.4f}")

## 6. Linear Regression

In [ ]:
from sklearn.linear_model import LinearRegression

# Prepare data
X = df_corr[['Height']]
y = df_corr['Weight']

# Fit model
model = LinearRegression()
model.fit(X, y)

print(f"Intercept: {model.intercept_:.4f}")
print(f"Slope: {model.coef_[0]:.4f}")
print(f"R-squared: {model.score(X, y):.4f}")

# Predictions
predictions = model.predict(X)
residuals = y - predictions

print(f"\nResiduals (first 5): {residuals.values}")
print(f"Mean squared error: {(residuals**2).mean():.4f}")

## 7. Confidence Intervals

In [ ]:
# Bootstrap confidence interval
def bootstrap_ci(data, n_iterations=1000, ci=95):
    bootstrap_means = []
    for _ in range(n_iterations):
        sample = np.random.choice(data, size=len(data), replace=True)
        bootstrap_means.append(sample.mean())
    
    lower = np.percentile(bootstrap_means, (100 - ci) / 2)
    upper = np.percentile(bootstrap_means, 100 - (100 - ci) / 2)
    return lower, upper

# Calculate bootstrap CI
lower_ci, upper_ci = bootstrap_ci(df['Score'].values)
print(f"Bootstrap 95% CI for mean:")
print(f"Lower: {lower_ci:.2f}, Upper: {upper_ci:.2f}")
print(f"Sample mean: {df['Score'].mean():.2f}")

In [ ]:
# Standard error and confidence interval (parametric)
mean = df['Score'].mean()
std = df['Score'].std()
n = len(df['Score'])
se = std / np.sqrt(n)

# 95% CI using t-distribution
confidence = 0.95
alpha = 1 - confidence
t_critical = stats.t.ppf(1 - alpha/2, df=n-1)

margin_of_error = t_critical * se
ci_lower = mean - margin_of_error
ci_upper = mean + margin_of_error

print(f"\nParametric 95% CI for mean:")
print(f"Mean: {mean:.2f}")
print(f"Standard Error: {se:.4f}")
print(f"95% CI: [{ci_lower:.2f}, {ci_upper:.2f}]")